


Contributions:

Part 1. Main Contributor text: Lovro Antic s235263
    - Subcontributor: Uffe Grøn Roepstorff s245109


Part 2. Main Contributor code and text: Oskar Karlsson s244771
    -Subcontributor: Lovro Antic s235263

Part 3, Main Contributor code and text.: Uffe Grøn Roepstorff s245109
    -Subcontributor: Oskar Karlsson s244771

Link to GitHub: https://github.com/oggefaderen/CompSci

Note that this link leads you to the group's general GitHub repository so that you can see the individual commits. To access Assignment1, go to the folder 'Assignment1'. Here you find the final Jupyter notebook, Assignment1.ipynb, together with the individual files we worked in before compiling it into the final Jupyter notebook.


In [42]:
import requests
import pandas as pd
import time
from tqdm import tqdm
import unicodedata
import re
from bs4 import BeautifulSoup
from joblib import Parallel, delayed

## Part 1


#### **Pros and Cons of the Custom-made Vs Ready-made Data**

*What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book*

Centola – Custom-made data (experiment):
The experiment was purposely designed to study behavioral contagion, allowing full control over network structure, participant assignment, and treatment conditions. This minimizes confounding variables and enables causal inference. However, the artificial online environment may not reflect how behaviors spread on real world platforms, limiting external validity. Additionally, custom-made data collection is resource intensive and relies on a self selected sample of health interested participants, introducing selection bias.

Nicolaides – Ready-made data (observational):
Data was passively collected from a fitness tracking app with over one million users, providing a large, diverse, real world sample across long time periods, this strengthens the statistical power. However, the data was not collected for research purposes, introducing potential sample bias (only app users are represented), unobserved confounders, and limited control over variables such as weather or social context.




#### **Different Influence**

*How do you think these differences can influence the interpretation of the results in each study?*



Centola's Study: The controlled environment allowed for causal claims on how complex contagion drives behaviorial spread. With random assignmet to either clustered, or random networks, the observed difference in adoption can be directly attributed to the network structure.
Although there is a causal claim, the study could lack external validity.

Nicolaides's Study: This offers natural validity, with the data coming from natural interactions. While the study shows a correlation of 'contagious running' even when accounting intrumental variable setup wit the weather, it has a weaker causal claim than Centolas study.



# Part 2

### Retrieve data

In [ ]:
EMAIL_1 = 'oskar@dasu.dk'
EMAIL_2 = 'ofk@adhocracy.dk'
BASE_URL_AUTHORS = 'https://api.openalex.org/authors'
BASE_URL_WORKS = 'https://api.openalex.org/works'

""" We use 2 API keys for part 2, as searching the 1500 authors uses more than all the credits in one key """
KEY_1 = 'NPOnvy5MTAFvbY53ruvGnT' # Other account ket: 'YU1NMyYLC0SJYghyehfP0i'
KEY_2 = 'LEj3qajHn5Do2VxtWsguka' # Other account

In [44]:
df = pd.read_csv('authors.csv') # list of authors from week 1

names = set() # identify unique authors in the author dataset
for entry in df['author'].dropna():
    for name in entry.split(','):
        name = name.strip()
        if name:
            names.add(name)

names = list(names)

print(f'Total unique authors: {len(names)}')

Total unique authors: 1565


### Helper functions, created iteratively after encountering problems

In [45]:
def normalize_name(name):
    # Normalize the name to remove accents and special characters
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    return name

def fallback_name(name):
    # Remove middle initials like "A." or "B.C."
    return re.sub(r'\b[A-Z]\.\s*', '', name).strip()

### Search each name in OpenAlex and retrieve information

In [46]:
# Initialize lists to store author information
author_ids = []
display_names = []
works_api_urls = []
h_indexes = []
works_counts = []
country_codes = []

key_idx = 0
KEYS = [KEY_1, KEY_2]

for name in tqdm(names):
    params = {
        'search': normalize_name(name),
        'api_key': KEYS[key_idx % len(KEYS)],
        'per_page': 1,
    }
    r = requests.get('https://api.openalex.org/authors', params=params)
    if r.status_code == 429: # Handle rate limit by switching API keys
        key_idx += 1
        params['api_key'] = KEYS[key_idx % len(KEYS)]
        r = requests.get('https://api.openalex.org/authors', params=params)
    result = r.json().get('results', []) if r.status_code == 200 else []

    if not result:
        retry_name = fallback_name(normalize_name(name))
        if retry_name != normalize_name(name):
            params['search'] = retry_name
            r = requests.get('https://api.openalex.org/authors', params=params)
            if r.status_code == 429:
                key_idx += 1
                params['api_key'] = KEYS[key_idx % len(KEYS)]
                r = requests.get('https://api.openalex.org/authors', params=params)
            result = r.json().get('results', []) if r.status_code == 200 else []


    if result:
        author = result[0] # Take the first result as the most relevant match, could be improved with better matching logic
        author_ids.append(result[0]['id'])
        display_names.append(result[0]['display_name'])
        works_api_urls.append(result[0]['works_api_url'])
        h_indexes.append(author.get('summary_stats', {}).get('h_index')) # Inside summary_stats dict
        works_counts.append(result[0]['works_count'])
        # Get country code from last known institution
        institution = author.get('last_known_institutions', [])
        country_code = institution[0].get('country_code') if institution else None
        country_codes.append(country_code)
    else:
        author_ids.append(None)
        display_names.append(None)
        works_api_urls.append(None)
        h_indexes.append(None)
        works_counts.append(None)
        country_codes.append(None)

100%|██████████| 1565/1565 [08:28<00:00,  3.08it/s]


### Store in dataframe

In [47]:
# Store the collected data in a DataFrame
authors_df = pd.DataFrame({
    'author_id': author_ids,
    'display_name': display_names,
    'works_api_url': works_api_urls,
    'h_index': h_indexes,
    'works_count': works_counts,
    'country_code': country_codes
})

# strip url from author_id to get the clean actual id
authors_df['author_id'] = authors_df['author_id'].str.replace('https://openalex.org/', '', regex=False)

In [48]:
# Remove rows with missing author_id and save to csv
authors_df = authors_df.dropna(subset=['author_id'])
authors_df.to_csv('part1_authors_info.csv', index=False)
authors_df

,author_id,display_name,works_api_url,h_index,works_count,country_code
0,A5069479167,Didem Gündoğdu,https://api.openalex.org/works?filter=author.i...,3.0,10.0,DE
1,A5037727403,Siming Huang,https://api.openalex.org/works?filter=author.i...,38.0,115.0,US
2,A5022660208,Maria Brandén,https://api.openalex.org/works?filter=author.i...,20.0,63.0,SE
3,A5078658990,Yann Renisio,https://api.openalex.org/works?filter=author.i...,4.0,30.0,FR
4,A5044898565,Dino Pedreschi,https://api.openalex.org/works?filter=author.i...,58.0,386.0,IT
...,...,...,...,...,...,...
1559,A5109210308,Wilbur A. Franklin,https://api.openalex.org/works?filter=author.i...,92.0,637.0,None
1560,A5100608518,Chan Young Park,https://api.openalex.org/works?filter=author.i...,37.0,112.0,KR
1561,A5013327023,Munjung Kim,https://api.openalex.org/works?filter=author.i...,2.0,5.0,None
1563,A5004308290,Ece Calikus,https://api.openalex.org/works?filter=author.i...,6.0,14.0,SE


### Reflection questions
Which challenges did you encounter? How did you address them?
We encountered several challenges while identifying the authors in OpenAlex, and here is a few examples:
1. Without proper queries you spend a lot more credits than given. We adressed it by running tests on smaller batches of the dataset, still statistically representative of the dataset. To stay within rate limits, we needed to use two keys. The script automatically switches key when exceeded. Filtering is 10x cheaper but requires a lot more finetuning to obtain same result as searching - so we used search.
2. Not all the information we were to retrieve could be gathered directly from the API response. We resolved this by e.g. retrieving the author's last known institution and extracting the country code from it. We also had issues finding the h-index, but by inspecting our output dictionaries carefully, we found it nested inside a sub-dictionary called summary_stats.

Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data?

Not all authors in the csv file could be found in OpenAlex by just making an API-call with their name. This was the main problem in this exercise and we solved it by created a set of helper functions. The first called normalize_name strips special characters from the name, e.g. Étienne --> Etienne. The second function called fallback_name removes middle initials from a name - names are not necesarily spelled with middle initial in OpenAlex. By implementing these two function, we reduced the percentage of missing authors in OpenAlex from 55% to 39%. We chose this approach because it is simple and interpretable. However, it may introduce false positives, for instance, stripping a middle initial could match the wrong author if two researchers share a first and last name.

# Part 3 

In [ ]:
""" For stability, we use a 3rd api key, to ensure we don't run out of credits """

API_KEY = 'U2HIfMm7lzzOACIZH2GvPS' # Alt key: 'I7BU1yoA12r5RaYCvba8eC' # When we run out of keys

### Work Count filter

In [50]:
#There is no works count filter in the filtes, so we filter the works count before
#we find their works

mask = (authors_df['works_count'] > 5) & (authors_df['works_count'] < 5000)
authors_df = authors_df[mask]

### Works finder

In [ ]:
WORKING_URL = 'https://api.openalex.org/works'
EMAIL = "s245109@dtu.dk"  

#Defining topics.
social_ids = "C144024400|C15744967|C162324750|C17744445"
quant_ids = "C33923547|C121332964|C41008148"
BATCH_SIZE = 25

ids = authors_df['author_id'].tolist() #
#Creating a list of all our Batches
batches = [ids[i : i + BATCH_SIZE] for i in range(0, len(ids), BATCH_SIZE)]

#Creating workers.
def fetch_works_for_batch(batch_ids):
    batch_idstr = '|'.join(batch_ids)
    current_cursor = '*'
    batch_works = [] 
    
    filter_string = (
        f'author.id:{batch_idstr},'
        'authors_count:<10,'
        f'concepts.id:{social_ids},'
        f'concepts.id:{quant_ids},'
        'cited_by_count:>10'
    )

    while True:
        params = {
            'filter': filter_string,
            'select': 'id,publication_year,cited_by_count,authorships,title,abstract_inverted_index,concepts',
            'per_page': 200,
            'cursor': current_cursor,
            'mailto': EMAIL,
            'api_key': API_KEY 
        }

        try:
            response = requests.get(WORKING_URL, params=params)
            
            #Checking that we recieved the call
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                meta = data.get('meta', {})
                
                if not results:
                    break
                    
                for work in results:
                    author_ids = [
                        auth['author']['id'].split('/')[-1] if auth.get('author') and auth['author'].get('id')
                        else auth['author']['id']
                        for auth in work.get('authorships', [])
                    ]
                    batch_works.append({
                        'id': work.get('id'),
                        'publication_year': work.get('publication_year'),
                        'cited_by_count': work.get('cited_by_count'),
                        'title': work.get('title'),
                        'author_ids': author_ids, 
                        'abstract_inverted_index': work.get('abstract_inverted_index')
                    })
                #Curser updates
                current_cursor = meta.get('next_cursor')
                if not current_cursor:
                    break
                    
            elif response.status_code == 429:
                # If we hit the API request limit.
                print('sleeping for 2 seconds.')
                time.sleep(2)
                continue 
            else:
                #Errors log.
                print(f"ERROR: {response.status_code}")
                break
                
        except Exception as e:
            #Backup for the work.
            print('sleeping for 2 seconds.')
            time.sleep(2)
            continue

    #Returning workds
    return batch_works

# 3. Kør Joblib Parallel processen
print(f"Working with {len(batches)} batches")

#Retrieving n=4 jobs. 
results_nested = Parallel(n_jobs=4)(
    delayed(fetch_works_for_batch)(batch) for batch in tqdm(batches, desc="Collecting works")
)

# Flattening the list
all_works = [work for sublist in results_nested for work in sublist]

print(f"\nRetrieved {len(all_works)} works total.")

Starter parallel download for 52 batches...



Retrieved 17269 works total.


### Creation and export of DataFrames

In [52]:
#Creating the dataframes
D1 = pd.DataFrame(all_works)
D1.to_csv('Ex3_get_all_works.csv')
print(D1.shape, f'we have found {D1.shape[0]} works') 
D2 = D1[['id', 'publication_year', 'cited_by_count', 'author_ids']]
D3 = D1[['id', 'title', 'abstract_inverted_index']]
D2.to_csv('Ex3_D2.csv')
D3.to_csv('Ex3_D3.csv')

(17269, 6) we have found 17269 works


#### Unique researchers count

In [ ]:
#Retrieving all the unique ID'
#Collecting all authors in a list
all_auth_ids = [author_id for sublist in D2['author_ids'] for author_id in sublist]

# Removing dupes via set.
unique_researchers = len(set(all_auth_ids))
print('Unique researchers found', unique_researchers)

Unique researchers found 24194


#### Dataset summaries:

##### **Dataset summary.** 
*How many works are listed in your Dataset D2 (IC2S2 papers) dataframe? How many unique researchers have co-authored these works?*




We have found **17269 works** with our filters,  as well a retrieveing **24194 unique authors**.
(Note, if there could be API erros, this nubmer could be a bit less.)

From 1525 researchers we managed to scrape from the IC2S2 conference, who have contributed to a total of 17269 works we find it reasonable that we have 
found 24194 authors in total. This is a large number, but we find it reasonable, our method just finds anyone who have contributed to a paper, which could be 
in the field of computational social science, as it still could fetch researcher from other fields, whom just contributed to one CompSci paper.


##### **Efficiency in code.** 

*Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?*


We optimized our code to maximize efficiency and minimize API requests using the following strategies:

1. Pre-filtering: Excluded authors outside the 5–5,000 works range before initiating any API calls.

2. Cursor Pagination: Utilized cursors to retrieve works efficiently per request.

3. Server-side Filtering: Applied complex API filters to download only relevant data.

4. Batching: Grouped 25 authors per request using the | (OR) operator, reducing total requests and conserving tokens.

5. Robustness: Implemented a pause condition to handle rate limits and prevent API overloads.

6. Parallel Processing: Executed requests concurrently across 4 cores. This step alone reduced the total execution time by a third.

Comparing the optimized work-retrieval with the unoptimized author retrieval, we see the major improvements. Retrieving 1490 authors in 8.28 minutes vs 17269 works in 49 seconds (on our computer). We thus see, that with a super optimized API search, we greatly improve the speed and price of retrieval.


#### **Filtering Criteria and Dataset Relevance** 
*Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?*

The filters we used help find relevant CSS works and researchers. Filtering works per author removes large institutions and irrelevant authors. Requiring 10+ citations helps us look only at influential work, although this limit could be arbitrary. Excluding projects with >10 authors mitigates megaprojects that are beyond the scope of just CSS research and could dominate. Concept IDs are also crucial, as there is no accurate CSS label, so works should cover both subjects.

One flaw of our method is that it prioritizes older, established researchers while excluding new researchers and PhDs, as they lack citations and output. There could also be many CSS papers labeled under only one field, e.g., only Economics. Lastly, more niche subfields could be skipped, as smaller fields have fewer citations.